# Bicycle Rental — Personal Accuracy Experiments
ฉบับส่วนตัว แยกจากงานทีม | IFJC4002

เปิดใน Google Colab แล้วเลือก **Runtime → Run all** ใช้ CPU ได้ ไม่ต้องเปิด GPU
รอบแรก Colab จะขอเชื่อม Drive ระยะเวลาขึ้นกับทรัพยากรที่ได้รับและจำนวนชุดพารามิเตอร์

ไฟล์นี้ทำครบตั้งแต่อ่าน CSV ตรวจข้อมูล แบ่งชุดตามเวลา ทดลอง baseline และโมเดลเดิม
จูน Random Forest และ HistGradientBoosting ด้วย TimeSeriesSplit เทียบ validation
ประเมิน test แบบระบุข้อจำกัด แล้วบันทึกโมเดล กราฟ ตาราง และ ZIP สำหรับนำไปใช้ต่อ

**สถานะไฟล์:** ยังไม่ได้รันการทดลอง ไม่มีคะแนนที่แต่งขึ้น ผลทั้งหมดจะคำนวณเมื่อคุณรัน
งานนี้เป็นการทดลองส่วนตัวที่มี AI ช่วยเตรียมโค้ด ไม่ได้ระบุว่าเป็นผลงานที่คุณเขียนเองทั้งหมด

โฟลเดอร์ผลลัพธ์: `MyDrive/Bicycle_Rental_Personal_Experiments_2026-10-28/runs/<run_id>`
ข้อมูลทีมถูกเปิดอ่านอย่างเดียว โค้ดนี้ไม่เขียนลงในโฟลเดอร์ทีม ไม่ push GitHub และไม่แก้โมเดลหลัก


## 1. ตั้งค่า
ใช้ข้อมูลอาจารย์ชุดเดิมเป็นค่าเริ่มต้น หากไม่พบ ให้แก้ `SOURCE_CSV` เป็น path ใน Drive ของคุณ
อย่าสุ่มข้อมูลใหม่เพื่อให้คะแนนดูดีขึ้น แบ่งตาม Guide: train 70%, validation 15%, test 15%

`SEARCH_ITERATIONS=12` หมายถึงลองพารามิเตอร์สุ่ม 12 ชุดต่อโมเดล ใช้ 4 folds ตามเวลา
เริ่มที่ค่านี้ก่อน หากจะเพิ่มจำนวน ให้ตัดสินใจก่อนดูผล test รอบใหม่
ชุด test เดิมเคยเปิดใช้แล้ว ผลที่ได้จึงเป็น repeated evaluation ไม่ใช่ผลจาก holdout ที่ไม่เคยเห็น


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, shutil, platform, importlib.metadata

PERSONAL_NAME = 'Bicycle_Rental_Personal_Experiments_2026-10-28'
PERSONAL_ROOT = Path('/content/drive/MyDrive') / PERSONAL_NAME
TEAM_ROOT = Path('/content/drive/MyDrive/Bicycle_Rental_Project_Team')
SOURCE_CSV = TEAM_ROOT / '00_SHARED/dataset_from_lecturer/03_Bicycle_Rental_Demand.csv'
SEARCH_ITERATIONS = 12
CV_SPLITS = 4
CV_TEST_HOURS = 168
SEARCH_JOBS = 2
SEED = 42
EVALUATE_REPEATED_TEST = True

assert PERSONAL_ROOT != TEAM_ROOT and TEAM_ROOT not in PERSONAL_ROOT.parents
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = PERSONAL_ROOT / 'runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
FIGURES = RUN_DIR / 'figures'
MODELS = RUN_DIR / 'model_bundle'
FIGURES.mkdir(); MODELS.mkdir()
def sha256(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Private output directory:', RUN_DIR)


## 2. ตรวจ runtime และอ่านข้อมูล
ใช้ไลบรารีที่ Colab มีอยู่ บันทึกรุ่นจริงไว้พร้อมผล ถ้าขาดแพ็กเกจ ให้ติดตั้งจาก PyPI ก่อนรันต่อ
ข้อมูลเป็น synthetic teaching data หนึ่งแถวแทนหนึ่งชั่วโมง เป้าหมายคือ `rental_count` หน่วย rentals/hour


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn, joblib
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

VERSIONS = {name: importlib.metadata.version(name) for name in ['scikit-learn','numpy','pandas','scipy','joblib','matplotlib']}
save_json('environment.json', dict(python=sys.version, platform=platform.platform(), packages=VERSIONS))
if not SOURCE_CSV.is_file():
    raise FileNotFoundError(f'ไม่พบ {SOURCE_CSV} เพิ่ม shortcut โฟลเดอร์ทีมใน MyDrive หรือแก้ SOURCE_CSV ให้ตรงกับ CSV ที่ได้รับ')
raw = pd.read_csv(SOURCE_CSV, parse_dates=['timestamp'])
FEATURES = ['hour','temperature_c','humidity_pct','weather','wind_speed_kmh','is_working_day','day_of_week']
TARGET = 'rental_count'
required = ['record_id','timestamp'] + FEATURES + [TARGET]
assert not (set(required) - set(raw.columns)), 'Missing expected dataset columns'
assert len(raw) == 2000, 'Expected the same 2,000-record teaching dataset; review before adapting'
assert raw.timestamp.notna().all() and raw.timestamp.is_unique
raw = raw.sort_values('timestamp').reset_index(drop=True)
assert raw.timestamp.diff().dropna().eq(pd.Timedelta(hours=1)).all(), 'CV assumes successive hourly observations'
assert raw[TARGET].notna().all() and np.isfinite(raw[TARGET]).all() and (raw[TARGET] >= 0).all()
assert raw.hour.eq(raw.timestamp.dt.hour).all()
assert raw.day_of_week.eq(raw.timestamp.dt.dayofweek).all()
assert raw.is_working_day.eq((raw.day_of_week < 5).astype(int)).all()
assert set(raw.weather.dropna()) <= {'Clear','Cloudy','Light_Rain','Heavy_Rain'}
assert raw.hour.between(0,23).all() and raw.day_of_week.between(0,6).all()
assert raw.humidity_pct.dropna().between(35,99).all()
assert (raw.wind_speed_kmh.dropna() >= 0).all()
DATA_SHA = sha256(SOURCE_CSV)
shutil.copy2(SOURCE_CSV, RUN_DIR / 'dataset_snapshot.csv')
save_json('data_quality.json', dict(rows=len(raw), columns=list(raw.columns), missing=raw.isna().sum().to_dict(), duplicate_rows=int(raw.duplicated().sum()), source=str(SOURCE_CSV), source_sha256=DATA_SHA, data_type='synthetic teaching data'))
print('Dataset checked:', len(raw), 'rows; SHA256:', DATA_SHA)
display(raw[FEATURES].head())


## 3. แบ่งตามเวลา และสำรวจเฉพาะ train
ใช้ 1,400 แถวแรกฝึก ถัดมา 300 แถวเลือกโมเดล และ 300 แถวสุดท้ายประเมินภายหลัง
กราฟในช่วงนี้ใช้ train เพื่อไม่ให้การออกแบบ feature พึ่งข้อมูล test


In [ ]:
n = len(raw)
train = raw.iloc[:int(n*.70)].copy()
validation = raw.iloc[int(n*.70):int(n*.85)].copy()
test_reserved = raw.iloc[int(n*.85):].copy()
assert train.timestamp.max() < validation.timestamp.min() < test_reserved.timestamp.min()
X_train, y_train = train[FEATURES], train[TARGET]
X_validation, y_validation = validation[FEATURES], validation[TARGET]
manifest = dict(dataset_sha256=DATA_SHA, features=FEATURES, excluded=['record_id','timestamp',TARGET], partitions={k:dict(n=len(v),start=str(v.timestamp.min()),end=str(v.timestamp.max())) for k,v in [('train',train),('validation',validation),('test',test_reserved)]})
save_json('split_manifest.json',manifest)
fig, ax = plt.subplots(1,2,figsize=(12,4))
train[TARGET].hist(bins=30, ax=ax[0]); ax[0].set(title='Training rental counts',xlabel='rentals/hour',ylabel='Hours')
train.groupby(['hour','is_working_day'])[TARGET].mean().unstack().plot(ax=ax[1])
ax[1].set(title='Training mean demand by hour',ylabel='rentals/hour')
fig.tight_layout(); fig.savefig(FIGURES/'training_eda.png',dpi=160); plt.show()


## 4. สร้าง feature และ pipeline
ทดสอบชั่วโมง/วันแบบ sin–cos และช่วงเดินทางในวันทำงาน โดยใช้เฉพาะข้อมูลที่รู้ก่อนทำนาย
ไม่ใช้ยอดเช่าย้อนหลังในรอบนี้ เพราะจะเปลี่ยนข้อมูลที่แอปต้องรับ
ทุก imputer, scaler และ encoder อยู่ใน pipeline จึง fit ใหม่บน training fold แต่ละรอบ
บันทึกฟังก์ชัน feature เป็นไฟล์ Python เพื่อให้โหลดโมเดลข้าม runtime ได้


In [ ]:
FEATURE_MODULE_SOURCE = "import numpy as np\nFEATURES = ['hour','temperature_c','humidity_pct','weather','wind_speed_kmh','is_working_day','day_of_week']\nEXTRA = ['hour_sin','hour_cos','hour_sin2','hour_cos2','weekday_sin','weekday_cos','morning_commute','evening_commute']\ndef add_features(frame):\n    if not hasattr(frame, 'columns'):\n        raise TypeError('Pass a pandas DataFrame with named raw input columns')\n    missing=set(FEATURES)-set(frame.columns)\n    if missing:\n        raise ValueError('Missing raw inputs: '+', '.join(sorted(missing)))\n    data=frame[FEATURES].copy()\n    hour=data['hour']; weekday=data['day_of_week']; work=data['is_working_day']\n    data['hour_sin']=np.sin(2*np.pi*hour/24)\n    data['hour_cos']=np.cos(2*np.pi*hour/24)\n    data['hour_sin2']=np.sin(4*np.pi*hour/24)\n    data['hour_cos2']=np.cos(4*np.pi*hour/24)\n    data['weekday_sin']=np.sin(2*np.pi*weekday/7)\n    data['weekday_cos']=np.cos(2*np.pi*weekday/7)\n    data['morning_commute']=hour.between(7,9).astype(int)*work\n    data['evening_commute']=hour.between(17,19).astype(int)*work\n    return data\n"
(RUN_DIR/'personal_features.py').write_text(FEATURE_MODULE_SOURCE,encoding='utf-8')
sys.path.insert(0,str(RUN_DIR))
import personal_features
from personal_features import add_features, EXTRA
NUMERIC = [c for c in FEATURES if c!='weather']
def make_pipeline(estimator, engineered=False):
    numeric = NUMERIC + (EXTRA if engineered else [])
    preprocessing=ColumnTransformer([
        ('numeric',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),numeric),
        ('weather',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),['weather'])],remainder='drop')
    return Pipeline([('features',FunctionTransformer(add_features,validate=False) if engineered else 'passthrough'),('preprocess',preprocessing),('model',estimator)])
cv=TimeSeriesSplit(n_splits=CV_SPLITS,test_size=CV_TEST_HOURS,gap=0)
fold_rows=[]
for fold,(fit_idx,eval_idx) in enumerate(cv.split(X_train),1):
    assert fit_idx.max() < eval_idx.min()
    fold_rows.append(dict(fold=fold,train_rows=len(fit_idx),evaluation_rows=len(eval_idx),train_end=str(train.iloc[fit_idx[-1]].timestamp),evaluation_start=str(train.iloc[eval_idx[0]].timestamp)))
pd.DataFrame(fold_rows).to_csv(RUN_DIR/'cv_folds.csv',index=False)
display(pd.DataFrame(fold_rows))


## 5. วัด baseline และโมเดลเดิมใหม่
เปรียบเทียบบนข้อมูลและ split รอบเดียวกัน ไม่ใช้คะแนนเก่าแทนผลของรอบนี้
Random Forest เดิมใช้ 100 ต้น seed 42 ส่วน Linear Regression เป็นอีกอัลกอริทึมที่ใช้เทียบ


In [ ]:
def regression_metrics(y,prediction):
    prediction=np.asarray(prediction)
    assert np.isfinite(prediction).all()
    return dict(MAE=float(mean_absolute_error(y,prediction)),RMSE=float(np.sqrt(mean_squared_error(y,prediction))),R2=float(r2_score(y,prediction)))
models={}
records=[]
reference_models={
    'Mean baseline':make_pipeline(DummyRegressor(strategy='mean')),
    'Linear Regression':make_pipeline(LinearRegression()),
    'Current RF (100 trees)':make_pipeline(RandomForestRegressor(n_estimators=100,random_state=SEED,n_jobs=1))}
for name,pipeline in reference_models.items():
    cv_mae=-cross_val_score(pipeline,X_train,y_train,cv=cv,scoring='neg_mean_absolute_error',n_jobs=SEARCH_JOBS,error_score='raise')
    pipeline.fit(X_train,y_train)
    models[name]=pipeline
    score=regression_metrics(y_validation,pipeline.predict(X_validation))
    records.append(dict(candidate=name,CV_MAE_mean=float(cv_mae.mean()),CV_MAE_std=float(cv_mae.std()),**score))
    print(name,score)
pd.DataFrame(records).to_csv(RUN_DIR/'reference_validation.csv',index=False)


## 6. จูน Random Forest และ Gradient Boosting
ค้นหาด้วย training folds เท่านั้น ใช้ค่า MAE เฉลี่ยต่ำสุดใน TimeSeriesSplit เลือกพารามิเตอร์
Gradient Boosting ปิด early stopping ที่แบ่งข้อมูลภายในแบบสุ่ม เพื่อให้การประเมินตามเวลาชัดเจน
ทุกพารามิเตอร์และคะแนน CV จะถูกบันทึก แม้โมเดลที่จูนแล้วจะแพ้โมเดลเดิม


In [ ]:
search_specs={
    'Tuned RF + time features':(
        make_pipeline(RandomForestRegressor(random_state=SEED,n_jobs=1),engineered=True),
        {'model__n_estimators':[100,200,400],'model__max_depth':[None,6,10,16],
         'model__min_samples_leaf':[1,2,4,8],'model__max_features':[1.0,0.7,'sqrt']}),
    'Tuned HistGradientBoosting + time features':(
        make_pipeline(HistGradientBoostingRegressor(random_state=SEED,early_stopping=False),engineered=True),
        {'model__max_iter':[150,300,500],'model__learning_rate':[0.03,0.06,0.1],
         'model__max_leaf_nodes':[7,15,31],'model__min_samples_leaf':[10,20,35],
         'model__l2_regularization':[0.0,1.0,5.0]})}
search_summary={}
for index,(name,(pipeline,parameters)) in enumerate(search_specs.items(),1):
    print('Searching:',name,flush=True)
    search=RandomizedSearchCV(pipeline,parameters,n_iter=SEARCH_ITERATIONS,scoring='neg_mean_absolute_error',cv=cv,random_state=SEED,n_jobs=SEARCH_JOBS,refit=True,verbose=1,error_score='raise',return_train_score=True)
    search.fit(X_train,y_train)
    models[name]=search.best_estimator_
    score=regression_metrics(y_validation,search.predict(X_validation))
    records.append(dict(candidate=name,CV_MAE_mean=float(-search.best_score_),CV_MAE_std=float(search.cv_results_['std_test_score'][search.best_index_]),**score))
    pd.DataFrame(search.cv_results_).to_csv(RUN_DIR/f'search_{index}_cv_results.csv',index=False)
    search_summary[name]=dict(best_parameters=search.best_params_,best_CV_MAE=float(-search.best_score_),validation=score)
    save_json('search_summary.json',search_summary)
    print('Best parameters:',search.best_params_,'Validation:',score,flush=True)


## 7. เลือกโมเดลจาก validation และวิเคราะห์ error
เลือก MAE ต่ำสุด ถ้าเท่ากันให้ดู RMSE แล้วใช้ชื่อตัดสินเพื่อให้ทำซ้ำได้
ยังไม่เปิดคะแนน test ตัดสินใจว่าจะใช้โมเดลใด ถ้าจูนแล้วไม่ดีขึ้นก็เก็บโมเดลเดิมไว้
คะแนน CV ของชุดที่ค้นหามี selection bias จึงไม่ใช้เป็นคะแนนทดสอบสุดท้าย


In [ ]:
comparison=pd.DataFrame(records).sort_values(['MAE','RMSE','candidate']).reset_index(drop=True)
comparison.to_csv(RUN_DIR/'validation_comparison.csv',index=False)
display(comparison)
winner_name=comparison.iloc[0].candidate
winner=models[winner_name]
current_mae=float(comparison.loc[comparison.candidate=='Current RF (100 trees)','MAE'].iloc[0])
winner_mae=float(comparison.iloc[0].MAE)
improvement_pct=100*(current_mae-winner_mae)/current_mae if current_mae else 0.0
selection=dict(candidate=winner_name,metric='validation MAE',tie_break=['RMSE','candidate name'],selected_before_test=True,best_validation_MAE=winner_mae,current_RF_validation_MAE=current_mae,validation_improvement_percent=improvement_pct,refit_policy='train only; preserve fitted selected estimator',synthetic_data=True)
save_json('model_selection.json',selection)
print('Selected:',winner_name)
print(f'Validation MAE improvement over current RF: {improvement_pct:.2f}%')
validation_predictions=winner.predict(X_validation)
errors=validation[['timestamp','hour','weather','is_working_day',TARGET]].copy()
errors['prediction']=validation_predictions
errors['absolute_error']=np.abs(errors[TARGET]-errors.prediction)
errors.to_csv(RUN_DIR/'validation_predictions.csv',index=False)
for group in ['hour','weather','is_working_day']:
    summary=errors.groupby(group).absolute_error.agg(['count','mean']).rename(columns={'mean':'MAE'})
    summary.to_csv(RUN_DIR/f'validation_errors_by_{group}.csv')
    display(summary)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].scatter(y_validation,validation_predictions,alpha=.4)
axes[0].set(xlabel='Actual rentals/hour',ylabel='Predicted rentals/hour',title='Validation: actual vs predicted')
axes[1].plot(errors.timestamp,errors[TARGET],label='Actual',alpha=.7)
axes[1].plot(errors.timestamp,errors.prediction,label='Predicted',alpha=.7)
axes[1].legend(); axes[1].set_title('Validation timeline')
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig(FIGURES/'validation_predictions.png',dpi=160); plt.show()


## 8. บันทึกโมเดลที่เลือก แล้วประเมิน test ซ้ำแบบเปิดเผย
ขั้นนี้ใช้โมเดลที่ล็อกแล้ว ไม่มีการจูนจากคะแนน test และไม่ refit ด้วย validation
หาก `EVALUATE_REPEATED_TEST=False` จะข้ามการคำนวณ test
ถ้ารัน cell นี้ซ้ำใน run เดิม จะอ่านผลที่บันทึกไว้แทน และตรวจ hash ให้ตรงกัน
ผลจากข้อมูลใหม่ที่ยังไม่เคยเห็นต้องแยกเก็บเป็นการประเมินอิสระในอนาคต


In [ ]:
shutil.copy2(RUN_DIR/'personal_features.py',MODELS/'personal_features.py')
MODEL_PATH=MODELS/'personal_best_pipeline.joblib'
if not MODEL_PATH.exists():
    joblib.dump(winner,MODEL_PATH,compress=3)
MODEL_SHA=sha256(MODEL_PATH)
loaded=joblib.load(MODEL_PATH)
np.testing.assert_allclose(loaded.predict(X_validation.iloc[:8]),winner.predict(X_validation.iloc[:8]),rtol=0,atol=1e-10)
test_record_path=RUN_DIR/'repeated_test_metrics.json'
if EVALUATE_REPEATED_TEST:
    if test_record_path.exists():
        test_record=json.loads(test_record_path.read_text(encoding='utf-8'))
        assert test_record['model_sha256']==MODEL_SHA
        print('Using existing test record for this exact artifact.')
    else:
        test_record=dict(candidate=winner_name,dataset_split='test',n=len(test_reserved),**regression_metrics(test_reserved[TARGET],loaded.predict(test_reserved[FEATURES])),model_sha256=MODEL_SHA,evaluation_status='REPEATED evaluation: this teaching test partition was inspected in earlier team work; not an untouched holdout',used_for_tuning=False)
        save_json('repeated_test_metrics.json',test_record)
    display(pd.DataFrame([test_record]))
else:
    test_record=None
    print('Test evaluation skipped. Validation-only results saved.')
metadata=dict(selected_model=winner_name,raw_features=FEATURES,excluded=['record_id','timestamp',TARGET],target=TARGET,unit='rentals/hour',model_sha256=MODEL_SHA,dataset_sha256=DATA_SHA,versions=VERSIONS,run_id=RUN_ID,fit_partition='train',postprocessing='none; raw continuous predictions',test_evaluation='repeated, not untouched',personal_experiment=True)
(MODELS/'metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
(MODELS/'requirements.txt').write_text('\n'.join(f'{name}=={version}' for name,version in VERSIONS.items())+'\n',encoding='utf-8')
save_json('run_manifest.json',dict(selection=selection,metadata=metadata,search_iterations=SEARCH_ITERATIONS,cv_folds=fold_rows,source_csv=str(SOURCE_CSV),outputs=str(RUN_DIR)))


## 9. ลองทำนายข้อมูลใหม่ และตรวจ input
ตัวอย่างนี้ใช้ข้อมูลอากาศที่ทราบก่อนชั่วโมงที่จะทำนาย ผลเป็นค่าเฉลี่ยประมาณการที่อาจมีทศนิยม
ตรวจหน่วยและช่วงที่รองรับก่อนนำไปใช้งาน โมเดลนี้ยังไม่ได้รับรองความแม่นยำในบริการจริง


In [ ]:
def validate_example(values):
    if set(values)!=set(FEATURES): raise ValueError('Pass exactly the seven raw input fields')
    for name in FEATURES:
        if name=='weather': continue
        value=values[name]
        if isinstance(value,bool) or not isinstance(value,(int,float)) or not np.isfinite(value):
            raise ValueError(f'{name}: finite numeric value required')
    for name,lo,hi in [('hour',0,23),('day_of_week',0,6),('is_working_day',0,1)]:
        if values[name]!=int(values[name]) or not lo<=values[name]<=hi: raise ValueError(name)
    if values['weather'] not in ['Clear','Cloudy','Light_Rain','Heavy_Rain']: raise ValueError('weather')
    if not 35<=values['humidity_pct']<=99: raise ValueError('humidity_pct')
    if values['wind_speed_kmh']<0: raise ValueError('wind_speed_kmh')
    if values['is_working_day']!=int(values['day_of_week']<5): raise ValueError('weekday/working-day inconsistency')
    return pd.DataFrame([values],columns=FEATURES)
example=dict(hour=8,temperature_c=26.0,humidity_pct=75.0,weather='Clear',wind_speed_kmh=10.0,is_working_day=1,day_of_week=2)
example_frame=validate_example(example)
prediction=float(loaded.predict(example_frame)[0])
assert np.isfinite(prediction)
print(f'Prediction: {prediction:.2f} rentals/hour')
checks=[]
for name,updates in [('invalid hour',{'hour':24}),('unknown weather',{'weather':'Snow'}),('nonfinite temperature',{'temperature_c':float('nan')}),('negative wind',{'wind_speed_kmh':-1}),('inconsistent weekday',{'day_of_week':6})]:
    try:
        validate_example({**example,**updates})
        passed=False
    except ValueError:
        passed=True
    checks.append(dict(case=name,passed=passed))
checks.append(dict(case='saved model parity',passed=bool(np.allclose(loaded.predict(example_frame),winner.predict(example_frame)))))
check_table=pd.DataFrame(checks)
check_table.to_csv(RUN_DIR/'inference_checks.csv',index=False)
assert check_table.passed.all()
display(check_table)


## 10. สรุปและส่งออก
ZIP มีโมเดล ฟังก์ชัน feature metadata และ requirements ส่วนกราฟ/ผลทดลองอยู่ใน run folder
หากจะโหลดโมเดลใน runtime ใหม่ ต้องใส่โฟลเดอร์ที่มี `personal_features.py` ใน `sys.path` ก่อน `joblib.load`
อย่าแทนที่ไฟล์ทีมโดยอัตโนมัติ ให้ตรวจผล validation ความสม่ำเสมอของ CV และข้อจำกัดก่อน


In [ ]:
bundle_readme="Personal bicycle-demand experiment\n\nThis model is separate from the team model. Synthetic teaching data only.\nTest scores are repeated evaluation after earlier test exposure.\nTo load: install the recorded dependencies, add this folder to sys.path,\nimport personal_features, then joblib.load('personal_best_pipeline.joblib').\nPass a pandas DataFrame with the seven raw input fields from metadata.json.\nNo clipping or rounding was applied when computing metrics.\n"
(MODELS/'README.txt').write_text(bundle_readme,encoding='utf-8')
archive=shutil.make_archive(str(RUN_DIR/'personal_model_bundle'),'zip',MODELS)
summary=(f'Personal experiment: {RUN_ID}\nSelected: {winner_name}\n'
         f'Validation MAE improvement over current RF: {improvement_pct:.2f}%\n'
         f'Validation metrics:\n{comparison.to_string(index=False)}\n'
         f'Repeated test result: {json.dumps(test_record,ensure_ascii=False)}\n'
         'No automatic update to the team model or GitHub.\n')
(RUN_DIR/'SUMMARY.txt').write_text(summary,encoding='utf-8')
print(summary)
print('All results:',RUN_DIR)
print('Model bundle:',archive)


## แหล่งอ้างอิงและขอบเขต
- ชีต Guide ใน `03_Bicycle_Rental_Demand.xlsx` ของอาจารย์: ข้อมูลจำลองรายชั่วโมงและ chronological split 70/15/15
- [TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)
- [RandomizedSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html)
- [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)
- [HistGradientBoostingRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)

ถ้าโมเดลใหม่ได้ validation ดีกว่า แปลว่าดีขึ้นบนช่วงเวลาที่ใช้เลือกโมเดลนี้ ยังไม่ใช่หลักฐานว่าดีกว่าในทุกช่วงเวลา
การใช้ validation เลือกระหว่างหลายตัวเลือกทำให้คะแนนมีความเอนเอียงได้ การยืนยันอิสระต้องใช้ข้อมูลใหม่
